# CoT-ENEM — Fase 3 no Colab

Antes de executar: **Ambiente de execução → Alterar tipo de ambiente de execução → GPU**. O notebook usa Qwen2.5-7B-Instruct em 4 bits, grava cada registro no Drive e retoma sem duplicar registros.

In [ ]:
# Edite somente estes valores.
REPOSITORY_URL = "https://github.com/SEU_USUARIO/cot-enem.git"
BRANCH = "main"
LIMIT = None  # None processa toda a base; use 1 primeiro para validar.
DRIVE_ROOT = "/content/drive/MyDrive/cot-enem"


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
for folder in ("data/processed", "outputs/datasets", "outputs/logs", "cache/huggingface"):
    (Path(DRIVE_ROOT) / folder).mkdir(parents=True, exist_ok=True)


In [ ]:
import os, shutil, subprocess
if not Path("/content").is_dir() or Path("/kaggle").exists():
    raise RuntimeError("Este notebook deve ser executado no Google Colab, não no Kaggle.")
PROJECT = Path("/content/cot-enem").resolve()
if PROJECT != Path("/content/cot-enem"):
    raise RuntimeError(f"Caminho de projeto inseguro: {PROJECT}")
if PROJECT.exists():
    shutil.rmtree(PROJECT)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPOSITORY_URL, str(PROJECT)], check=True)
os.chdir(PROJECT)
os.environ["HF_HOME"] = f"{DRIVE_ROOT}/cache/huggingface"
os.environ["TRANSFORMERS_CACHE"] = f"{DRIVE_ROOT}/cache/huggingface"
subprocess.run(["python", "-m", "pip", "install", "-q", "-e", ".[colab]"], check=True)


In [ ]:
# Confirma GPU e dependências antes de baixar/carregar o modelo.
subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["python", "-m", "cot_enem.cli", "verify", "--config", "configs/colab.yaml"], check=True)


Envie uma única vez o arquivo local `data/processed/enem_normalized.jsonl` para `Meu Drive/cot-enem/data/processed/`. A célula abaixo interrompe cedo se o arquivo não estiver lá.

In [ ]:
INPUT = Path(DRIVE_ROOT) / "data/processed/enem_normalized.jsonl"
OUTPUT = Path(DRIVE_ROOT) / "outputs/datasets/cot_enem_v1.jsonl"
if not INPUT.exists():
    raise FileNotFoundError(f"Envie enem_normalized.jsonl para: {INPUT}")
print(f"Entrada: {INPUT}")
print(f"Saída persistente: {OUTPUT}")


In [ ]:
command = [
    "python", "-m", "cot_enem.cli", "evolve",
    "--config", "configs/colab.yaml",
    "--strategies", "specify",
    "--input", str(INPUT),
    "--output", str(OUTPUT),
]
if LIMIT is not None:
    command += ["--limit", str(LIMIT)]
subprocess.run(command, check=True)


In [ ]:
import json
lines = [line for line in OUTPUT.read_text(encoding="utf-8").splitlines() if line.strip()]
print(f"Registros persistidos: {len(lines)}")
if lines:
    print(json.dumps(json.loads(lines[-1]), ensure_ascii=False, indent=2)[:4000])
